In [1]:
%load_ext autoreload
%autoreload 2

from compressible_core import chemistry_types, chemistry_utils
from compressible_core import constants


In [4]:
json_path = "/home/hhoechter/tum/jaxfluids_internship/data/species.json"
equilibrium_enthalpy_json = "/home/hhoechter/tum/jaxfluids_internship/data/equilibrium_enthalpy_fits.json"

my_species = chemistry_utils.load_species_from_gnoffo(general_data_path=json_path, equilibrium_enthalpy=equilibrium_enthalpy_json)

In [3]:
from jaxtyping import Array, Float
import jax 
from compressible_core import constants


def compute_equilibrium_enthalpy_polynomial(
    T_V: Float[Array, " N"],
    T_limit_low: Float[Array, "n_species n_ranges"],
    T_limit_high: Float[Array, "n_species n_ranges"],
    parameters: Float[Array, "n_species n_ranges n_parameters"],
    molar_masses: Float[Array, " n_species"],
) -> Float[Array, "n_species N"]:
    """Compute equilibrium enthalpy for all species using polynomial curve fits.

    This function computes specific enthalpy h(T) for all species simultaneously
    using temperature-dependent polynomial fits from NASA or similar databases.

    Args:
        T_V: Temperature array [K], shape (N,)
        T_limit_low: Lower temperature bounds for each range [K], shape (n_species, n_ranges)
        T_limit_high: Upper temperature bounds for each range [K], shape (n_species, n_ranges)
        parameters: Polynomial coefficients for each range, shape (n_species, n_ranges, n_parameters)
        molar_masses: Molecular mass for each species [kg/mol], shape (n_species,)

    Returns:
        Specific enthalpy [J/kg] for all species, shape (n_species, N)

    Notes:
        - Uses vmap to vectorize over species dimension
        - Polynomial form: h = (R/M) * (a1*T + a2*T^2/2 + ... + a6)
        - Temperature ranges are checked to select correct polynomial coefficients
    """

    n_ranges = T_limit_low.shape[1]

    def h_single_species(
        T_low: Float[Array, " n_ranges"],
        T_high: Float[Array, " n_ranges"],
        params: Float[Array, "n_ranges n_parameters"],
        M: float,
    ) -> Float[Array, " N"]:
        """Compute enthalpy for one species across all temperatures."""
        h_V = jnp.zeros_like(T_V)

        # Loop over temperature ranges (small fixed number, JAX will unroll)
        for i in range(n_ranges):
            # Mask for temperatures in this range
            mask = (T_V >= T_low[i]) & (T_V < T_high[i])

            # Extract polynomial coefficients for this range
            a = params[i, :]

            # Evaluate polynomial: h = (R/M) * (a0*T + a1*T^2/2 + ... + a5)
            h_range = (
                constants.R_universal
                / M
                * (
                    a[0] * T_V**1 / 1
                    + a[1] * T_V**2 / 2
                    + a[2] * T_V**3 / 3
                    + a[3] * T_V**4 / 4
                    + a[4] * T_V**5 / 5
                    + a[5]
                )
            )

            # Update h_V only where mask is True
            h_V = jnp.where(mask, h_range, h_V)

        return h_V

    # Vectorize over species dimension (axis 0 of each input)
    h_vectorized = jax.vmap(h_single_species, in_axes=(0, 0, 0, 0))

    return h_vectorized(T_limit_low, T_limit_high, parameters, molar_masses)

In [11]:
%load_ext autoreload
%autoreload 2

import jax
import jax.numpy as jnp
import plotly.graph_objects as go
from pathlib import Path

from compressible_core import thermodynamic_relations
from compressible_core.chemistry_utils import load_species_table_from_gnoffo

jax.config.update("jax_enable_x64", True)

import sys
from pathlib import Path

species_name = "N2"  # enthalpy of this species will be plotted

# Add src directory to path
repo_root = Path.cwd().parent  # Assuming notebook is in experiments/
src_path = repo_root / "src"
sys.path.insert(0, str(src_path))

# Load data
data_dir = Path("../data")
general_data = str(data_dir / "air_5_gnoffo.json")
enthalpy_data = str(data_dir / "air_5_gnoffo_equilibrium_enthalpy.json")

species_table = load_species_table_from_gnoffo(general_data, enthalpy_data)
# Get species index
species_index = species_table.names.index(species_name)

T = jnp.linspace(species_table.T_limit_low[species_index, 0], species_table.T_limit_high[species_index, -1], 35000, endpoint=False)

# Compute enthalpy
h = thermodynamic_relations.compute_e_vibrational(T_V=T, species_table=species_table)

h_species = h[species_index, :] / 1e6

# Plot with Plotly
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=T,
        y=h_species,
        mode="lines",
        name=f"{species_name} (Enthalpy)",
        line=dict(color="blue", width=2),
    )
)
# Mark the temperature ranges
T_ranges_low = species_table.T_limit_low[species_index]
for T_range in T_ranges_low[1:]:  # Skip first one
    fig.add_vline(x=T_range, line_dash="dash", line_color="red", opacity=0.3)

fig.update_layout(
    title=f"Enthalpy vs Temperature for {species_name}",
    xaxis_title="Temperature [K]",
    yaxis_title="Enthalpy [MJ/kg]",
    hovermode="x unified",
    template="plotly_white",
)

fig.show()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
species_table.T_limit_low

Array([[  300,  1000,  6000, 15000, 25000],
       [  300,  1000,  6000, 15000, 25000],
       [  300,  1000,  6000, 15000, 25000],
       [  300,  1000,  6000, 15000, 25000],
       [  300,  1000,  6000, 15000, 25000]], dtype=int64)